In [1]:
# Core data manipulation libraries
import pandas as pd
import numpy as np

# Garbage collector for memory management
import gc

# System specific parameters and memory profiling
import os
import psutil

# Function to monitor current RAM usage
def print_memory_usage():
    process = psutil.Process(os.getpid())
    memory_megabytes = process.memory_info().rss / (1024 * 1024)
    print(f"Current RAM usage: {memory_megabytes:.2f} MB")

# Initial memory check before loading any data
print_memory_usage()

Current RAM usage: 124.31 MB


In [4]:
import pandas as pd
import numpy as np
import psutil
import os

# Function to monitor current RAM usage
def print_memory_usage():
    process = psutil.Process(os.getpid())
    memory_megabytes = process.memory_info().rss / (1024 * 1024)
    print(f"Current RAM usage: {memory_megabytes:.2f} MB")

# Function to downcast data types to the smallest possible format
def reduce_mem_usage(df):
    start_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage of dataframe is {:.2f} MB'.format(start_mem))
    
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
        else:
            # Convert strings to category type to save RAM
            df[col] = df[col].astype('category')

    end_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage after optimization is: {:.2f} MB'.format(end_mem))
    print('Decreased by {:.1f}%\n'.format(100 * (start_mem - end_mem) / start_mem))
    
    return df

# Define data path
data_path = './data/ashrae-energy-prediction/'

# 1. Load and compress building metadata
print("--- Loading and compressing building_metadata.csv ---")
building_df = pd.read_csv(data_path + 'building_metadata.csv')
building_df = reduce_mem_usage(building_df)

# 2. Load and compress weather data
print("--- Loading and compressing weather_train.csv ---")
weather_df = pd.read_csv(data_path + 'weather_train.csv')
weather_df = reduce_mem_usage(weather_df)

# 3. Load and compress the 20M rows dataset
print("--- Loading and compressing train.csv ---")
train_df = pd.read_csv(data_path + 'train.csv')
train_df = reduce_mem_usage(train_df)

# Final memory check after loading
print("--- Final Memory Check ---")
print_memory_usage()

--- Loading and compressing building_metadata.csv ---
Memory usage of dataframe is 0.07 MB
Memory usage after optimization is: 0.02 MB
Decreased by 73.9%

--- Loading and compressing weather_train.csv ---
Memory usage of dataframe is 9.60 MB
Memory usage after optimization is: 2.59 MB
Decreased by 73.1%

--- Loading and compressing train.csv ---
Memory usage of dataframe is 616.95 MB
Memory usage after optimization is: 173.84 MB
Decreased by 71.8%

--- Final Memory Check ---
Current RAM usage: 2321.41 MB


In [5]:
import gc

print("--- 1. Merging Train and Building Metadata ---")
# Left join train_df and building_df on 'building_id'
train_df = train_df.merge(building_df, on='building_id', how='left')

# Memory management: Delete building_df and force garbage collection
del building_df
gc.collect()
print("Merge 1 complete. building_metadata removed from RAM.")

print("--- 2. Merging with Weather Data ---")
# Left join train_df and weather_df on 'site_id' and 'timestamp'
train_df = train_df.merge(weather_df, on=['site_id', 'timestamp'], how='left')

# Memory management: Delete weather_df and force garbage collection
del weather_df
gc.collect()
print("Merge 2 complete. weather_train removed from RAM.")

print("--- 3. Final Memory Check After Merging ---")
# Display dataset structure and current RAM usage
train_df.info()
print_memory_usage()

--- 1. Merging Train and Building Metadata ---
Merge 1 complete. building_metadata removed from RAM.
--- 2. Merging with Weather Data ---
Merge 2 complete. weather_train removed from RAM.
--- 3. Final Memory Check After Merging ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20216100 entries, 0 to 20216099
Data columns (total 16 columns):
 #   Column              Dtype   
---  ------              -----   
 0   building_id         int16   
 1   meter               int8    
 2   timestamp           category
 3   meter_reading       float32 
 4   site_id             int8    
 5   primary_use         category
 6   square_feet         int32   
 7   year_built          float16 
 8   floor_count         float16 
 9   air_temperature     float16 
 10  cloud_coverage      float16 
 11  dew_temperature     float16 
 12  precip_depth_1_hr   float16 
 13  sea_level_pressure  float16 
 14  wind_direction      float16 
 15  wind_speed          float16 
dtypes: category(2), float16(9), float32(

In [6]:
import gc

print("--- 1. Converting Timestamp and Sorting ---")
# Convert timestamp back to datetime format for time-series operations
train_df['timestamp'] = pd.to_datetime(train_df['timestamp'])

# Sort values chronologically by site_id to prepare for accurate interpolation
train_df = train_df.sort_values(by=['site_id', 'timestamp'])
print("Timestamp conversion and sorting complete.")

print("--- 2. Interpolating Missing Weather Data ---")
# Define weather columns that need cleaning
weather_cols = ['air_temperature', 'cloud_coverage', 'dew_temperature', 
                'precip_depth_1_hr', 'sea_level_pressure', 'wind_direction', 'wind_speed']

# Apply linear interpolation grouped by site_id to respect geographic boundaries
for col in weather_cols:
    train_df[col] = train_df.groupby('site_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    
# Fallback: Fill any completely empty weather columns for specific sites with the global median
for col in weather_cols:
    if train_df[col].isnull().any():
        train_df[col] = train_df[col].fillna(train_df[col].median())

print("Weather data cleaning complete.")

print("--- 3. Missing Values Check and Memory ---")
# Verify that weather columns have 0 missing values
print("Missing values in weather columns:")
print(train_df[weather_cols].isnull().sum())

print_memory_usage()
gc.collect()

--- 1. Converting Timestamp and Sorting ---
Timestamp conversion and sorting complete.
--- 2. Interpolating Missing Weather Data ---
Weather data cleaning complete.
--- 3. Missing Values Check and Memory ---
Missing values in weather columns:
air_temperature       0
cloud_coverage        0
dew_temperature       0
precip_depth_1_hr     0
sea_level_pressure    0
wind_direction        0
wind_speed            0
dtype: int64
Current RAM usage: 3216.16 MB


0